In [ ]:
import os, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# --- 1. Taxonomy & Constants ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN, STD = (0.2860,), (0.3530,)

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
STRUCTURAL_GROUPS = {
    'Upper Body': [0, 2, 4, 6],
    'Lower & Full Body': [1, 3],
    'Footwear': [5, 7, 9],
    'Accessories': [8]
}
IDX_TO_STRUCTURAL = {idx: grp for grp, indices in STRUCTURAL_GROUPS.items() for idx in indices}

# --- 2. kefth/fashion-mnist ResNet18 Skeleton ---
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        return self.relu(out + residual)

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.avgpool = nn.AvgPool2d(7)
        self.fc = nn.Linear(256 * block.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        x = self.avgpool(x).view(x.size(0), -1)
        return self.fc(x)

# --- 3. Load Checkpoint (.pth) & Test Data ---
import zipfile
import glob

# Point this to your uploaded file (can be .zip or .pth)
UPLOADED_FILE = 'resnet18_fashionmnist.pth.zip'  # or 'kefth_resnet18_fmnist.pth'

# 1. If it's an outer .zip archive containing a .pth/.pt file, extract it automatically
if UPLOADED_FILE.endswith('.zip'):
    with zipfile.ZipFile(UPLOADED_FILE, 'r') as z:
        # Check if this is PyTorch's internal zip structure (contains data.pkl) or an outer zip
        names = z.namelist()
        if any(name.endswith('data.pkl') for name in names):
            # It's already a native PyTorch file that just had a .zip extension added!
            PTH_PATH = UPLOADED_FILE
        else:
            # It's a compressed archive containing a .pth file -> extract it
            z.extractall('./extracted_model')
            found_pths = glob.glob('./extracted_model/**/*.pth', recursive=True) + \
                         glob.glob('./extracted_model/**/*.pt', recursive=True)
            PTH_PATH = found_pths[0]
            print(f"Extracted '{PTH_PATH}' from zip archive.")
else:
    PTH_PATH = UPLOADED_FILE

# 2. Load across devices safely (GPU -> CPU, MPS -> CUDA, etc.)
model = ResNet(BasicBlock, [2, 2, 2, 2], num_classes=10).to(device)
checkpoint = torch.load(PTH_PATH, map_location=device, weights_only=False)

if isinstance(checkpoint, nn.Module):
    model = checkpoint.to(device)
else:
    # Unwrap common checkpoint dictionary keys ('state_dict', 'model', 'net', etc.)
    if isinstance(checkpoint, dict):
        for key in ['state_dict', 'model_state_dict', 'model', 'net']:
            if key in checkpoint:
                checkpoint = checkpoint[key]
                break
    # Strip 'module.' prefix if trained with DataParallel on the other device
    state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
    model.load_state_dict(state_dict)

model.eval()
print(f"Successfully loaded weights from '{PTH_PATH}' onto {device}!")

Successfully loaded weights from 'resnet18_fashionmnist.pth.zip' onto cpu!


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Ensure constants and normalization parameters exist
MEAN, STD = (0.2860,), (0.3530,)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load the test dataset
# raw_test_dataset provides unnormalized [0, 1] images for visual display and slider perturbations
raw_test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# eval_dataset provides normalized images for standard model inference
eval_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])
)

# 3. Compute baseline black-box predictions across all 10,000 test samples
eval_loader = DataLoader(eval_dataset, batch_size=256, shuffle=False)
model.eval()

all_probs, all_targets = [], []
with torch.no_grad():
    for imgs, labels in eval_loader:
        logits = model(imgs.to(device))
        p = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(p)
        all_targets.append(labels.numpy())

probs = np.vstack(all_probs)
targets = np.concatenate(all_targets)

print(f"Setup complete: {len(raw_test_dataset):,} test images loaded.")
print(f"Baseline Test Accuracy: {(probs.argmax(axis=1) == targets).mean() * 100:.2f}%")

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 306kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.64MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.3MB/s]


Setup complete: 10,000 test images loaded.
Baseline Test Accuracy: 94.22%


In [5]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.patches as patches

# Ensure custom widget manager is enabled if running in Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

class BlackBoxStressTester:
    """
    Phase 1 Interactive 'What-If' Stress-Tester & Adversary Canvas.
    Treats `model` strictly as a black box: Input Image -> Output Probabilities P(Y|X).
    """
    def __init__(self, model, raw_dataset, base_probs, base_targets, device):
        self.model = model.eval()
        self.raw_dataset = raw_dataset
        self.base_probs = base_probs
        self.base_targets = base_targets
        self.base_preds = base_probs.argmax(axis=1)
        self.device = device

        # Precompute baseline top-1 confidence and margin (p1 - p2)
        sorted_p = np.sort(base_probs, axis=1)[:, ::-1]
        self.base_conf = sorted_p[:, 0]
        self.base_margin = sorted_p[:, 0] - sorted_p[:, 1]

        self._build_ui()
        self._update_filtered_indices()
        self._render()

    @torch.no_grad()
    def _query_blackbox(self, raw_img_tensor):
        """Strict black-box query: normalizes [1, 28, 28] tensor in [0, 1] and returns [10] softmax probs."""
        norm_img = TF.normalize(raw_img_tensor, MEAN, STD).unsqueeze(0).to(self.device)
        logits = self.model(norm_img)
        return F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    def _build_ui(self):
        style = {'description_width': '115px'}
        layout = widgets.Layout(width='310px')

        # --- 1. Sample Selection Controls ---
        self.class_dropdown = widgets.Dropdown(
            options=[('All Classes', -1)] + [(f"{i}: {name}", i) for i, name in enumerate(CLASS_NAMES)],
            value=6, # Default to 'Shirt' (most ambiguous class)
            description='True Class:', style=style, layout=layout
        )
        self.filter_dropdown = widgets.Dropdown(
            options=[
                ('All Samples', 'all'),
                ('Correct Only', 'correct'),
                ('Misclassified Only', 'error'),
                ('Cliffhangers (Margin < 0.25)', 'cliffhanger')
            ],
            value='all', description='Filter Mode:', style=style, layout=layout
        )
        self.sample_slider = widgets.IntSlider(
            value=0, min=0, max=100, step=1,
            description='Sample #:', continuous_update=False, style=style, layout=layout
        )

        # --- 2. Geometric & Photometric Sliders ---
        self.rot_slider = widgets.FloatSlider(
            value=0.0, min=-45.0, max=45.0, step=2.5,
            description='Rotate (°):', continuous_update=False, style=style, layout=layout
        )
        self.scale_slider = widgets.FloatSlider(
            value=1.0, min=0.5, max=1.5, step=0.05,
            description='Scale (Zoom):', continuous_update=False, style=style, layout=layout
        )
        self.tx_slider = widgets.IntSlider(
            value=0, min=-6, max=6, step=1,
            description='Shift X (px):', continuous_update=False, style=style, layout=layout
        )
        self.ty_slider = widgets.IntSlider(
            value=0, min=-6, max=6, step=1,
            description='Shift Y (px):', continuous_update=False, style=style, layout=layout
        )
        self.contrast_slider = widgets.FloatSlider(
            value=1.0, min=0.2, max=2.5, step=0.1,
            description='Contrast:', continuous_update=False, style=style, layout=layout
        )
        self.bright_slider = widgets.FloatSlider(
            value=1.0, min=0.3, max=2.0, step=0.1,
            description='Brightness:', continuous_update=False, style=style, layout=layout
        )
        self.noise_slider = widgets.FloatSlider(
            value=0.0, min=0.0, max=0.40, step=0.02,
            description='Noise (σ):', continuous_update=False, style=style, layout=layout
        )
        self.hflip_check = widgets.Checkbox(
            value=False, description='Horizontal Flip', indent=False, layout=widgets.Layout(width='150px')
        )

        # --- 3. Interactive Occlusion Patch Controls ---
        self.occ_enable = widgets.Checkbox(
            value=False, description='Enable Occlusion Patch', indent=False, layout=widgets.Layout(width='200px')
        )
        self.occ_size = widgets.IntSlider(
            value=6, min=2, max=12, step=1,
            description='Patch Size:', continuous_update=False, style=style, layout=layout
        )
        self.occ_x = widgets.IntSlider(
            value=11, min=0, max=26, step=1,
            description='Patch X:', continuous_update=False, style=style, layout=layout
        )
        self.occ_y = widgets.IntSlider(
            value=6, min=0, max=26, step=1,
            description='Patch Y:', continuous_update=False, style=style, layout=layout
        )
        self.occ_val = widgets.FloatSlider(
            value=0.28, min=0.0, max=1.0, step=0.05,
            description='Patch Fill:', continuous_update=False, style=style, layout=layout
        )

        self.reset_btn = widgets.Button(
            description='Reset Perturbations', button_style='warning',
            icon='refresh', layout=widgets.Layout(width='310px', margin='8px 0px 0px 0px')
        )

        # Bind callbacks
        self.class_dropdown.observe(self._on_filter_change, names='value')
        self.filter_dropdown.observe(self._on_filter_change, names='value')
        self.ResetControls = [
            self.rot_slider, self.scale_slider, self.tx_slider, self.ty_slider,
            self.contrast_slider, self.bright_slider, self.noise_slider,
            self.hflip_check, self.occ_enable, self.occ_size, self.occ_x, self.occ_y, self.occ_val
        ]
        for w in [self.sample_slider] + self.ResetControls:
            w.observe(self._on_param_change, names='value')
        self.reset_btn.on_click(self._reset_perturbations)

        self.status_banner = widgets.HTML()
        self.plot_out = widgets.Output()

        # Assemble Control Sidebar
        col1 = widgets.VBox([
            widgets.HTML("<b>1. Dataset Explorer</b>"),
            self.class_dropdown, self.filter_dropdown, self.sample_slider,
            widgets.HTML("<hr style='margin:6px 0'><b>2. Geometric & Photometric</b>"),
            self.rot_slider, self.scale_slider, self.tx_slider, self.ty_slider,
            self.contrast_slider, self.bright_slider, self.noise_slider, self.hflip_check
        ], layout=widgets.Layout(padding='8px', border='1px solid #ddd', border_radius='6px'))

        col2 = widgets.VBox([
            widgets.HTML("<b>3. Occlusion Cursor (Feature Probe)</b>"),
            self.occ_enable, self.occ_size, self.occ_x, self.occ_y, self.occ_val,
            self.reset_btn,
            widgets.HTML("<hr style='margin:6px 0'><b>4. Live Boundary Alert</b>"),
            self.status_banner
        ], layout=widgets.Layout(padding='8px', border='1px solid #ddd', border_radius='6px'))

        self.ui = widgets.VBox([
            widgets.HBox([col1, col2], layout=widgets.Layout(gap='12px')),
            self.plot_out
        ])

    def _update_filtered_indices(self):
        c_idx = self.class_dropdown.value
        f_mode = self.filter_dropdown.value

        mask = np.ones(len(self.base_targets), dtype=bool)
        if c_idx != -1:
            mask &= (self.base_targets == c_idx)
        if f_mode == 'correct':
            mask &= (self.base_preds == self.base_targets)
        elif f_mode == 'error':
            mask &= (self.base_preds != self.base_targets)
        elif f_mode == 'cliffhanger':
            mask &= (self.base_margin < 0.25)

        self.matching_indices = np.where(mask)[0]
        if len(self.matching_indices) == 0:
            self.matching_indices = np.array([0])

        self.sample_slider.max = max(0, len(self.matching_indices) - 1)
        self.sample_slider.value = 0

    def _on_filter_change(self, change):
        self._update_filtered_indices()
        self._render()

    def _on_param_change(self, change):
        self._render()

    def _reset_perturbations(self, btn):
        # Temporarily unobserve to avoid 13 consecutive re-renders
        for w in self.ResetControls:
            w.unobserve(self._on_param_change, names='value')
        self.rot_slider.value = 0.0
        self.scale_slider.value = 1.0
        self.tx_slider.value = 0
        self.ty_slider.value = 0
        self.contrast_slider.value = 1.0
        self.bright_slider.value = 1.0
        self.noise_slider.value = 0.0
        self.hflip_check.value = False
        self.occ_enable.value = False
        self.occ_size.value = 6
        self.occ_x.value = 11
        self.occ_y.value = 6
        self.occ_val.value = 0.28
        for w in self.ResetControls:
            w.observe(self._on_param_change, names='value')
        self._render()

    def _apply_perturbations(self, raw_img):
        x = raw_img.clone()
        if self.hflip_check.value:
            x = TF.hflip(x)
        if (self.rot_slider.value != 0.0 or self.scale_slider.value != 1.0 or
            self.tx_slider.value != 0 or self.ty_slider.value != 0):
            x = TF.affine(
                x,
                angle=float(self.rot_slider.value),
                translate=[int(self.tx_slider.value), int(self.ty_slider.value)],
                scale=float(self.scale_slider.value),
                shear=0.0
            )
        if self.bright_slider.value != 1.0:
            x = TF.adjust_brightness(x, float(self.bright_slider.value))
        if self.contrast_slider.value != 1.0:
            x = TF.adjust_contrast(x, float(self.contrast_slider.value))
        if self.noise_slider.value > 0.0:
            torch.manual_seed(42) # Reproducible noise pattern while dragging
            x = x + torch.randn_like(x) * float(self.noise_slider.value)

        x = torch.clamp(x, 0.0, 1.0)

        # Apply Occlusion Patch if active
        occ_box = None
        if self.occ_enable.value:
            s = int(self.occ_size.value)
            ox = min(int(self.occ_x.value), 28 - s)
            oy = min(int(self.occ_y.value), 28 - s)
            x[:, oy:oy+s, ox:ox+s] = float(self.occ_val.value)
            occ_box = (ox, oy, s)

        return x, occ_box

    def _render(self):
        dataset_idx = int(self.matching_indices[self.sample_slider.value])
        raw_img, true_lbl = self.raw_dataset[dataset_idx]
        base_prob = self.base_probs[dataset_idx]
        base_pred = int(base_prob.argmax())

        pert_img, occ_box = self._apply_perturbations(raw_img)
        pert_prob = self._query_blackbox(pert_img)
        pert_pred = int(pert_prob.argmax())

        # Compute uncertainty stats
        s_base = np.sort(base_prob)[::-1]
        s_pert = np.sort(pert_prob)[::-1]
        base_margin = s_base[0] - s_base[1]
        pert_margin = s_pert[0] - s_pert[1]
        pert_entropy = -np.sum(pert_prob * np.log(np.clip(pert_prob, 1e-12, 1.0)))

        # Update HTML Status Banner
        true_name = CLASS_NAMES[true_lbl]
        base_name = CLASS_NAMES[base_pred]
        pert_name = CLASS_NAMES[pert_pred]
        true_grp = IDX_TO_STRUCTURAL[true_lbl]
        pert_grp = IDX_TO_STRUCTURAL[pert_pred]

        flipped = (pert_pred != base_pred)
        cross_group_err = (pert_grp != true_grp)

        if flipped:
            bg = "#fdecea" if pert_pred != true_lbl else "#e8f8f5"
            border = "#e74c3c" if pert_pred != true_lbl else "#2ecc71"
            headline = f"⚠️ PREDICTION FLIPPED: {base_name} → {pert_name}"
        else:
            bg = "#f8f9fa"
            border = "#bdc3c7"
            headline = f"Stable Prediction: {pert_name}"

        grp_badge = (
            f"<span style='color:#c0392b;font-weight:bold;'>Cross-Group Jump ({true_grp} → {pert_grp})</span>"
            if cross_group_err else f"<span style='color:#27ae60;'>Same Structural Group ({true_grp})</span>"
        )

        self.status_banner.value = f"""
        <div style='background:{bg}; border-left:5px solid {border}; padding:8px 10px; border-radius:4px; font-size:13px; width:290px;'>
            <b>{headline}</b><br>
            <b>Test Index:</b> #{dataset_idx} ({len(self.matching_indices)} match filter)<br>
            <b>Ground Truth:</b> {true_name} [{true_grp}]<br>
            <b>Group Containment:</b> {grp_badge}<br>
            <b>Confidence Δ:</b> {s_base[0]:.1%} → <b>{s_pert[0]:.1%}</b><br>
            <b>Margin (p1-p2):</b> {base_margin:.2f} → <b>{pert_margin:.2f}</b> | <b>Entropy:</b> {pert_entropy:.2f}
        </div>
        """

        with self.plot_out:
            clear_output(wait=True)
            fig, axes = plt.subplots(
                1, 3, figsize=(15.5, 4.5),
                gridspec_kw={'width_ratios': [1, 1, 1.8]}
            )

            # Panel 1: Original Unperturbed Input
            axes[0].imshow(raw_img.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
            axes[0].set_title(
                f"Baseline Input (#{dataset_idx})\nTrue: {true_name}\nBase Pred: {base_name} ({s_base[0]:.1%})",
                fontsize=11, fontweight='bold'
            )
            axes[0].axis('off')

            # Panel 2: Live Transformed + Occluded Input
            axes[1].imshow(pert_img.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
            if occ_box is not None:
                ox, oy, s = occ_box
                rect = patches.Rectangle(
                    (ox - 0.5, oy - 0.5), s, s,
                    linewidth=2, edgecolor='#e74c3c', facecolor='none', linestyle='--'
                )
                axes[1].add_patch(rect)
            title_color = '#c0392b' if pert_pred != true_lbl else '#27ae60'
            axes[1].set_title(
                f"Perturbed Black-Box Input\nNew Pred: {pert_name} ({s_pert[0]:.1%})\nMargin: {pert_margin:.2f} | Entropy: {pert_entropy:.2f}",
                fontsize=11, fontweight='bold', color=title_color
            )
            axes[1].axis('off')

            # Panel 3: Softmax Output Distribution Comparison (Baseline vs Perturbed)
            y_pos = np.arange(len(CLASS_NAMES))
            bar_h = 0.38
            axes[2].barh(y_pos + bar_h/2, base_prob, height=bar_h, color='#bdc3c7', label='Baseline P(Y|X)')

            pert_colors = []
            for i in range(len(CLASS_NAMES)):
                if i == true_lbl and i == pert_pred:
                    pert_colors.append('#2ecc71') # Green: True & Predicted
                elif i == pert_pred and i != true_lbl:
                    pert_colors.append('#e74c3c') # Red: Misclassified Top-1
                elif i == true_lbl:
                    pert_colors.append('#3498db') # Blue: Ground Truth (when missed)
                else:
                    pert_colors.append('#95a5a6') # Dark Gray: Other classes

            axes[2].barh(y_pos - bar_h/2, pert_prob, height=bar_h, color=pert_colors, label='Perturbed P(Y|X_pert)')
            axes[2].set_yticks(y_pos)
            axes[2].set_yticklabels([f"{name} [{IDX_TO_STRUCTURAL[i][:5]}.]" for i, name in enumerate(CLASS_NAMES)])
            axes[2].invert_yaxis()
            axes[2].set_xlim(0, 1.08)
            axes[2].set_xlabel("Black-Box Softmax Probability")
            axes[2].set_title("Live Output Probability Distribution P(Y|X)", fontsize=11, fontweight='bold')
            axes[2].legend(loc='lower right', frameon=True)

            for i, (bp, pp) in enumerate(zip(base_prob, pert_prob)):
                if pp > 0.04 or bp > 0.04:
                    axes[2].text(max(bp, pp) + 0.015, i, f"{pp:.2f}", va='center', fontsize=8.5, fontweight='bold')

            plt.tight_layout()
            plt.show()

    def display(self):
        display(self.ui)

# Launch the Interactive Black-Box Studio
studio = BlackBoxStressTester(model, raw_test_dataset, probs, targets, device)
studio.display()